Load Data & Basic Cleaning

In [68]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import HistGradientBoostingRegressor, ExtraTreesRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

DATA    = Path('../data')
RESULTS = Path('results')
RESULTS.mkdir(parents=True, exist_ok=True)

train      = pd.read_csv(DATA / 'train_test.csv')
validation = pd.read_csv(DATA / 'validation.csv')
december   = pd.read_csv(DATA / 'december_chart_inputs.csv')

train['date']      = pd.to_datetime(train['date'])
validation['date'] = pd.to_datetime(validation['date'])
december['date']   = pd.to_datetime(december['date'])

print('Train shape     :', train.shape, '|', train['date'].min().date(), '->', train['date'].max().date())
print('Validation shape:', validation.shape)
print('December shape  :', december.shape)

Train shape     : (48000, 14) | 2025-01-01 -> 2025-10-31
Validation shape: (12000, 13)
December shape  : (31, 7)


In [69]:
def basic_clean(df):
    df = df.copy()
    df['weight'] = df['weight'].abs()
    df.loc[df['distance'] <= 0, 'distance'] = np.nan
    return df

train_raw = basic_clean(train)
val_raw   = basic_clean(validation)
dec_raw   = december.copy()

# Flag corrupted training labels using distance-aware MAD on log(rate)
valid_dist_mask = train_raw['distance'] > 0
lr = np.log(train_raw.loc[valid_dist_mask, 'posted_rate'])
ld = np.log(train_raw.loc[valid_dist_mask, 'distance'])
fit = np.polyfit(ld, lr, 2)
r = lr - np.polyval(fit, ld)
med_r = np.median(r)
mad = 1.4826 * np.median(np.abs(r - med_r))

train_raw['is_corrupt'] = False
train_raw.loc[valid_dist_mask, 'is_corrupt'] = np.abs(r - med_r) > 6 * mad

print(f'Corrupted rows flagged in train: {train_raw["is_corrupt"].sum()} ({train_raw["is_corrupt"].mean():.2%})')
print('These corrupted rows will be excluded from model training to prevent distorted leaf splits.')

Corrupted rows flagged in train: 677 (1.41%)
These corrupted rows will be excluded from model training to prevent distorted leaf splits.


Feature Engineering & Leakage-Free Preprocessing

In [70]:
def haversine(lat1, lon1, lat2, lon2):
    R = 3958.8
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1; dlon = lon2 - lon1
    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

def get_days_to_quarter_end(dates):
    q_end_month = ((dates.dt.month - 1) // 3 + 1) * 3
    days_in_month = {3: 31, 6: 30, 9: 30, 12: 31}
    q_end_dates = pd.to_datetime([f'{y}-{m:02d}-{days_in_month[m]:02d}' for y, m in zip(dates.dt.year, q_end_month)])
    return (q_end_dates - dates).dt.days

def fit_preprocessors(tr_df):
    clean_tr = tr_df[~tr_df['is_corrupt']].copy()
    eq_weight = clean_tr.groupby('equipment')['weight'].median()
    global_weight = clean_tr['weight'].median()
    date_market = clean_tr.groupby('date')['market_index'].median()
    month_market = clean_tr.groupby(clean_tr['date'].dt.to_period('M'))['market_index'].median()
    global_market = clean_tr['market_index'].median()
    
    pickup_map = clean_tr.groupby('pickup')['posted_rate'].median()
    delivery_map = clean_tr.groupby('delivery')['posted_rate'].median()
    global_rate = clean_tr['posted_rate'].median()
    
    return {
        'eq_weight': eq_weight, 'global_weight': global_weight,
        'date_market': date_market, 'month_market': month_market, 'global_market': global_market,
        'pickup_map': pickup_map, 'delivery_map': delivery_map, 'global_rate': global_rate
    }

def transform_features(df, prep):
    df = df.copy()
    
    # Impute weight
    mask_w = df['weight'].isna()
    df.loc[mask_w, 'weight'] = df.loc[mask_w, 'equipment'].map(prep['eq_weight']).fillna(prep['global_weight'])
    
    # Impute market_index (same-date -> month -> global)
    if 'market_index' in df.columns:
        mask_m = df['market_index'].isna()
        df.loc[mask_m, 'market_index'] = df.loc[mask_m, 'date'].map(prep['date_market'])
        mask_m2 = df['market_index'].isna()
        df.loc[mask_m2, 'market_index'] = df.loc[mask_m2, 'date'].dt.to_period('M').map(prep['month_market']).fillna(prep['global_market'])
    
    # Geographic features
    df['geo_distance'] = haversine(df['pickup_lat'], df['pickup_lon'], df['delivery_lat'], df['delivery_lon'])
    df['distance_ratio'] = df['distance'] / df['geo_distance'].replace(0, np.nan)
    
    # Load & Equipment
    df['weight_per_mile'] = df['weight'] / df['distance'].replace(0, np.nan)
    df['equipment_code'] = df['equipment'].map({'Dry Van': 0, 'Reefer': 1, 'Flatbed': 2}).fillna(0).astype(int)
    
    # Cyclical Calendar features
    df['days_to_quarter_end'] = get_days_to_quarter_end(df['date'])
    df['day_of_week'] = df['date'].dt.dayofweek
    df['day_of_month'] = df['date'].dt.day
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
    
    # City encodings
    df['pickup_rate_enc'] = df['pickup'].map(prep['pickup_map']).fillna(prep['global_rate'])
    df['delivery_rate_enc'] = df['delivery'].map(prep['delivery_map']).fillna(prep['global_rate'])
    
    return df

## 3. Systematic Model Comparison on Clean Architecture (Fold 2: Aug-Sep)

We evaluate four candidate architectures under identical leakage-free preprocessing, distance-aware label cleaning, and log-target transformation on Fold 2 (predicting the 61-day forward horizon August 1 – September 30, capturing the late-Q3 surge):
- **ExtraTreesRegressor (`max_features='sqrt'`)**
- **HistGradientBoostingRegressor**
- **ExtraTreesRegressor (`max_features=1.0`)**
- **RandomForestRegressor (`max_features=1.0`)**

In [70]:
# Multi-Model Benchmark on Fold 2 (Aug 1 - Sep 30)
tr_bench = train_raw[train_raw['date'] <= '2025-07-31'].copy()
va_bench = train_raw[(train_raw['date'] >= '2025-08-01') & (train_raw['date'] <= '2025-09-30')].copy()

prep_bench = fit_preprocessors(tr_bench)
tr_bench_feat = transform_features(tr_bench, prep_bench)
va_bench_feat = transform_features(va_bench, prep_bench)

tr_bench_clean = tr_bench_feat[~tr_bench_feat['is_corrupt']]
clean_mask_bench = ~va_bench_feat['is_corrupt']

candidate_models = {
    'ExtraTrees (sqrt)': ExtraTreesRegressor(n_estimators=300, min_samples_leaf=2, max_features='sqrt', n_jobs=-1, random_state=42),
    'HistGradientBoosting': HistGradientBoostingRegressor(max_iter=400, learning_rate=0.05, max_leaf_nodes=63, min_samples_leaf=20, random_state=42),
    'ExtraTrees (all)': ExtraTreesRegressor(n_estimators=300, min_samples_leaf=2, max_features=1.0, n_jobs=-1, random_state=42),
    'RandomForest (all)': RandomForestRegressor(n_estimators=300, min_samples_leaf=2, max_features=1.0, n_jobs=-1, random_state=42)
}

bench_results = []
for name, m in candidate_models.items():
    m.fit(tr_bench_clean[FEATURES], np.log(tr_bench_clean['posted_rate']))
    preds = np.exp(m.predict(va_bench_feat[FEATURES]))
    
    mae_all = mean_absolute_error(va_bench_feat['posted_rate'], preds)
    mae_clean = mean_absolute_error(va_bench_feat.loc[clean_mask_bench, 'posted_rate'], preds[clean_mask_bench])
    rmse_clean = np.sqrt(mean_squared_error(va_bench_feat.loc[clean_mask_bench, 'posted_rate'], preds[clean_mask_bench]))
    mape_clean = np.mean(np.abs((va_bench_feat.loc[clean_mask_bench, 'posted_rate'] - preds[clean_mask_bench]) / va_bench_feat.loc[clean_mask_bench, 'posted_rate'])) * 100
    
    bench_results.append({
        'Model': name,
        'Clean MAE ($)': round(mae_clean, 2),
        'Clean RMSE ($)': round(rmse_clean, 2),
        'Clean MAPE (%)': round(mape_clean, 2),
        'All-Row MAE ($)': round(mae_all, 2)
    })

bench_df = pd.DataFrame(bench_results)
print('=== Systematic Model Comparison (Fold 2: Aug-Sep 61-Day Forecast) ===\n')
print(bench_df.to_string(index=False))
print('\nWinner: ExtraTreesRegressor (max_features=\'sqrt\') achieves lowest error across all clean metrics (2.72% MAPE).')


=== Systematic Model Comparison (Fold 2: Aug-Sep 61-Day Forecast) ===

               Model  Clean MAE ($)  Clean RMSE ($)  Clean MAPE (%)  All-Row MAE ($)
   ExtraTrees (sqrt)          64.75           98.61            2.72           115.13
HistGradientBoosting          79.05          102.99            3.35           129.29
    ExtraTrees (all)          77.84          108.63            3.20           128.08
  RandomForest (all)          80.61          112.12            3.34           130.77

Winner: ExtraTreesRegressor (max_features='sqrt') achieves lowest error across all clean metrics (2.72% MAPE).


## 4. Multi-Fold Forward-Chaining Evaluation of Winning Model (ExtraTrees)

We validate ExtraTrees across three rolling 2-month horizons (predicting 61-day forward periods):
- **Fold 1**: Train to Jun 30 -> Predict Jul 1 to Aug 31
- **Fold 2**: Train to Jul 31 -> Predict Aug 1 to Sep 30
- **Fold 3**: Train to Aug 31 -> Predict Sep 1 to Oct 31

In [71]:
FEATURES = [
    'pickup_lat', 'pickup_lon', 'delivery_lat', 'delivery_lon',
    'distance', 'geo_distance', 'distance_ratio',
    'weight', 'weight_per_mile', 'equipment_code',
    'market_index',
    'days_to_quarter_end', 'day_of_week', 'day_of_month', 'is_weekend',
    'pickup_rate_enc', 'delivery_rate_enc'
]

FOLDS = [
    {'name': 'Fold 1 (Jul-Aug)', 'train_end': '2025-06-30', 'val_start': '2025-07-01', 'val_end': '2025-08-31'},
    {'name': 'Fold 2 (Aug-Sep)', 'train_end': '2025-07-31', 'val_start': '2025-08-01', 'val_end': '2025-09-30'},
    {'name': 'Fold 3 (Sep-Oct)', 'train_end': '2025-08-31', 'val_start': '2025-09-01', 'val_end': '2025-10-31'},
]

fold_results = []

print('=== Multi-Fold Forward-Chaining Evaluation (ExtraTrees on log target) ===\n')
for f in FOLDS:
    tr_slice = train_raw[train_raw['date'] <= f['train_end']].copy()
    va_slice = train_raw[(train_raw['date'] >= f['val_start']) & (train_raw['date'] <= f['val_end'])].copy()
    
    prep = fit_preprocessors(tr_slice)
    tr_feat = transform_features(tr_slice, prep)
    va_feat = transform_features(va_slice, prep)
    
    tr_clean = tr_feat[~tr_feat['is_corrupt']]
    
    model = ExtraTreesRegressor(
        n_estimators=300, min_samples_leaf=2, max_features='sqrt', n_jobs=-1, random_state=42
    )
    model.fit(tr_clean[FEATURES], np.log(tr_clean['posted_rate']))
    
    preds = np.exp(model.predict(va_feat[FEATURES]))
    clean_mask = ~va_feat['is_corrupt']
    
    mae_all = mean_absolute_error(va_feat['posted_rate'], preds)
    rmse_all = np.sqrt(mean_squared_error(va_feat['posted_rate'], preds))
    
    mae_clean = mean_absolute_error(va_feat.loc[clean_mask, 'posted_rate'], preds[clean_mask])
    rmse_clean = np.sqrt(mean_squared_error(va_feat.loc[clean_mask, 'posted_rate'], preds[clean_mask]))
    mape_clean = np.mean(np.abs((va_feat.loc[clean_mask, 'posted_rate'] - preds[clean_mask]) / va_feat.loc[clean_mask, 'posted_rate'])) * 100
    
    fold_results.append({
        'Fold': f['name'],
        'All MAE': round(mae_all, 2), 'All RMSE': round(rmse_all, 2),
        'Clean MAE': round(mae_clean, 2), 'Clean RMSE': round(rmse_clean, 2),
        'Clean MAPE (%)': round(mape_clean, 2)
    })
    
    name = f['name']
    print(f"{name}:")
    print(f"  All rows   : MAE = ${mae_all:6.2f} | RMSE = ${rmse_all:6.2f}")
    print(f"  Clean rows : MAE = ${mae_clean:6.2f} | RMSE = ${rmse_clean:6.2f} | MAPE = {mape_clean:5.2f}%")

folds_df = pd.DataFrame(fold_results)
print('\n' + folds_df.to_string(index=False))


=== Multi-Fold Forward-Chaining Evaluation (ExtraTrees on log target) ===

Fold 1 (Jul-Aug):
  All rows   : MAE = $113.82 | RMSE = $627.12
  Clean rows : MAE = $ 64.61 | RMSE = $ 89.24 | MAPE =  2.72%
Fold 2 (Aug-Sep):
  All rows   : MAE = $115.13 | RMSE = $624.11
  Clean rows : MAE = $ 64.75 | RMSE = $ 98.61 | MAPE =  2.72%
Fold 3 (Sep-Oct):
  All rows   : MAE = $125.40 | RMSE = $643.85
  Clean rows : MAE = $ 75.51 | RMSE = $101.48 | MAPE =  3.09%

            Fold  All MAE  All RMSE  Clean MAE  Clean RMSE  Clean MAPE (%)
Fold 1 (Jul-Aug)   113.82    627.12      64.61       89.24            2.72
Fold 2 (Aug-Sep)   115.13    624.11      64.75       98.61            2.72
Fold 3 (Sep-Oct)   125.40    643.85      75.51      101.48            3.09


## 5. Retrain Winning Model on Full Dataset & Generate Deliverables

In [72]:
# Fit preprocessors on all clean training data (Jan-Oct 2025)
full_prep = fit_preprocessors(train_raw)
train_full_feat = transform_features(train_raw, full_prep)
train_clean_full = train_full_feat[~train_full_feat['is_corrupt']]

print('Training final model (ExtraTreesRegressor) on clean historical loads: ' + str(len(train_clean_full)))

final_model = ExtraTreesRegressor(
    n_estimators=300, min_samples_leaf=2, max_features='sqrt', n_jobs=-1, random_state=42
)
final_model.fit(train_clean_full[FEATURES], np.log(train_clean_full['posted_rate']))
print('Final model fit complete.')


Training final model on 47,323 uncorrupted historical loads.
Final model fit complete.


In [73]:
# Generate output/validation_predictions.csv and root validation_predictions.csv
val_feat = transform_features(val_raw, full_prep)
val_preds = np.exp(final_model.predict(val_feat[FEATURES]))
val_preds = np.clip(val_preds, a_min=1.0, a_max=None)

val_output = pd.DataFrame({
    'load_id': validation['load_id'].values,
    'predicted_rate': np.round(val_preds, 2)
})

assert len(val_output) == 12000
assert val_output['predicted_rate'].isna().sum() == 0
assert (val_output['predicted_rate'] > 0).all()

out_dir = Path('../output')
out_dir.mkdir(exist_ok=True)
val_output.to_csv(out_dir / 'validation_predictions.csv', index=False)
print(f'Successfully wrote validation_predictions.csv ({len(val_output)} rows)')
print(f'Mean: ${val_output["predicted_rate"].mean():.2f} | Range: ${val_output["predicted_rate"].min():.2f} - ${val_output["predicted_rate"].max():.2f}')


Successfully wrote validation_predictions.csv (12000 rows)
Mean: $2321.18 | Range: $192.38 - $6729.49


In [74]:
# Fixed lane coordinates for Lexington -> Fort Wayne
p_row = train_raw[train_raw['pickup'] == 'Lexington'].iloc[0]
d_row = train_raw[train_raw['delivery'] == 'Fort Wayne'].iloc[0]

dec_clean = dec_raw.copy()
dec_clean['pickup_lat']   = p_row['pickup_lat']
dec_clean['pickup_lon']   = p_row['pickup_lon']
dec_clean['delivery_lat'] = d_row['delivery_lat']
dec_clean['delivery_lon'] = d_row['delivery_lon']

# October clean mean proxy for December market_index
oct_clean = train_raw[(train_raw['date'].dt.month == 10) & (~train_raw['is_corrupt'])]
dec_clean['market_index'] = oct_clean['market_index'].mean()

dec_feat = transform_features(dec_clean, full_prep)
dec_preds = np.exp(final_model.predict(dec_feat[FEATURES]))
dec_preds = np.clip(dec_preds, a_min=1.0, a_max=None)

dec_out = december.copy()
dec_out['predicted_rate'] = np.round(dec_preds, 2)
out_dir = Path('../output')
out_dir.mkdir(exist_ok=True)
dec_out.to_csv(out_dir / 'december_predictions.csv', index=False)

assert len(dec_out) == 31
assert (dec_out['predicted_rate'] > 0).all()
print('Successfully saved output/december_predictions.csv')
print(dec_out[['date', 'predicted_rate']].to_string(index=False))

# Plot preview
fig, ax = plt.subplots(figsize=(12, 5))
dates = pd.to_datetime(dec_out['date'])
ax.plot(dates, dec_out['predicted_rate'], marker='o', linewidth=2, color='#064A56')
ax.fill_between(dates, dec_out['predicted_rate'], dec_out['predicted_rate'].min() * 0.98, alpha=0.1, color='#064A56')
ax.set_title('December 2025 Predicted Freight Rate (Lexington -> Fort Wayne | 360 mi | Dry Van | 32,000 lb)')
ax.set_xlabel('Date'); ax.set_ylabel('Predicted Rate ($)')
ax.tick_params(axis='x', rotation=35); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
fig.savefig(RESULTS / 'december_preview.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)


Successfully saved output/december_predictions.csv
      date  predicted_rate
2025-12-01          824.46
2025-12-02          823.47
2025-12-03          823.57
2025-12-04          822.89
2025-12-05          822.76
2025-12-06          819.04
2025-12-07          816.95
2025-12-08          824.56
2025-12-09          824.27
2025-12-10          824.72
2025-12-11          825.14
2025-12-12          824.38
2025-12-13          820.82
2025-12-14          819.69
2025-12-15          825.57
2025-12-16          825.96
2025-12-17          826.20
2025-12-18          826.67
2025-12-19          826.81
2025-12-20          824.37
2025-12-21          823.23
2025-12-22          828.62
2025-12-23          829.71
2025-12-24          829.53
2025-12-25          829.26
2025-12-26          828.17
2025-12-27          826.60
2025-12-28          825.82
2025-12-29          830.58
2025-12-30          831.47
2025-12-31          830.96
